In [0]:
# Databricks Notebook Source
dbutils.widgets.removeAll()

In [0]:
# DBTITLE 1, Define Production Pipeline Widgets
dbutils.widgets.text(
    "GoldConfigPath", 
    "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/FactGapsInCare/Config/factgapsincare.json", 
    "Target Fact Gold Config JSON Path"
)

# Extract config path directly from the widget
config_path = dbutils.widgets.get("GoldConfigPath")

In [0]:
# DBTITLE 2, Core Fact Layer Execution Function
def trigger_fact_processing(config_file_path: str):
    """Triggers the generalized Fact engine loop by passing the config file path."""
    # Production path mapping to the parent orchestration path structure
    ROOT_DIR = "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing"
    fact_notebook_path = f"{ROOT_DIR}/FactGapsInCare/Notebooks/GenericSubGroupProcessing"
    
    print(f"--> Invoking Fact Layer Engine...")
    print(f"    Target Config: {config_file_path}")
    print(f"    Target Notebook: {fact_notebook_path}")
    
    payload_arguments = {
        "SubGroupConfigPath": config_file_path
    }
    
    # Run the generic processing engine
    result = dbutils.notebook.run(fact_notebook_path, 1200, arguments=payload_arguments)
    print(f"    Fact Layer Response: {result}")
    return result

In [0]:
# DBTITLE 3, Main Pipeline Sequenced Execution
def main():
    print("=======================================================")
    print("STARTING LIFECYCLE FOR PIPELINE: FactGapsInCare_Gold")
    print("=======================================================")
    
    try:
        # Run downstream Fact parsing table transformations and merges
        pipeline_result = trigger_fact_processing(config_path)
        
        print("\n=======================================================")
        print("PIPELINE PROCESS COMPLETE")
        print(f"Final Outcome: {pipeline_result}")
        print("=======================================================")
        
        dbutils.notebook.exit(f"SUCCESS: {pipeline_result}")
        
    except Exception as e:
        error_msg = f"Pipeline execution faulted during processing: {str(e)}"
        print(f"CRITICAL: {error_msg}")
        dbutils.notebook.exit(f"FAILED: {error_msg}")


In [0]:
if __name__ == "__main__":
    main()